In [1]:
from  pyspark.sql import SparkSession
customer_watched_path = "UserWatched.csv"
users_path = "Users.csv"
episodes_path = "Episodes.csv"
tvseries_path = "TVSeries.csv"

In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import max, count

def ex_1(session:SparkSession):
#     Number of long TV series for each TV series genre. The first part of this application
# computes the number of long TV series for each TV series genre. A TV series is
# classified as a long TV series if it lasts more than 10 seasons (i.e. if it is characterized
# by more than 10 seasons). Store the result in the first HDFS output folder. Specifically,
# store one TV series genre per output line with its number of long TV series. Consider
# only the TV series genre with at least one long TV series.
# Output format of each output line (first part):
# TV series genre, Number of long TV series for this TV series genre
    episodes = session.read.load(episodes_path, format="csv", header = True, inferSchema = True)
    tvseries = session.read.load(tvseries_path, format="csv", header = True, inferSchema = True)
    max_seasons = episodes.groupBy("TVSID").agg(max("SeasonNumber").alias("MaxSeasons")).selectExpr("TVSID as TVSID2", "MaxSeasons")
    joinedDF = tvseries.join(max_seasons, tvseries.TVSID == max_seasons.TVSID2)
    joinedDF = joinedDF.select("TVGenre", "MaxSeasons").where("MaxSeasons >= 10")
    finalDF = joinedDF.groupBy("TVGenre").agg(count("MaxSeasons").alias("Number of long tv series"))
    max_seasons.show(10)
    joinedDF.show(10)
    finalDF.show(10)

def ex_2(session:SparkSession):
#     The longest TV series each user has started watching. For each user, the second part
# of this application selects the longest TV series in terms of number of seasons among
# those the user has started watching. A user has started watching a TV series if he/she
# has watched the first episode of the first season of that TV series (i.e., the episode of
# TV series with SeasonNumber=1 and EpisodeNumber=1). Analogously to the first part,
# the "length" of a TV series corresponds to the number of seasons of that TV series.
# Store the result in the second HDFS output folder. Specifically, there is one output line
# for each combination (username, TVSID of the longest TV series username has started
# watching).
# Output format of each output line (second part):
# Username, TVSID of the longest TV series username has started watching
    userWatched = session.read.load(customer_watched_path, format="csv", header = True, inferSchema = True)
    userWatched = userWatched.select("Username", "TVSID", "EpisodeNumber", "SeasonNumber").where("SeasonNumber = 1 and EpisodeNumber = 1")
    episodes = session.read.load(episodes_path, format="csv", header = True, inferSchema = True)
    max_seasons = episodes.groupBy("TVSID").agg(max("SeasonNumber").alias("MaxSeasons")).selectExpr("TVSID as TVSID2", "MaxSeasons")
    joinedDF = userWatched.join(max_seasons, max_seasons.TVSID2 == userWatched.TVSID)

    joinedDF = joinedDF.select("Username", "TVSID","MaxSeasons").groupBy("Username","TVSID").agg(max("MaxSeasons").alias("MaxSeasons"))
    finalDF = joinedDF.groupBy("Username").agg(max("MaxSeasons").alias("MaxSeasons2")).selectExpr("Username as Username2", "MaxSeasons2")

    solved = joinedDF.join(finalDF, joinedDF.Username == finalDF.Username2)
    solved = solved.select("Username", "TVSID").where("MaxSeasons = MaxSeasons2")
    userWatched.show()
    max_seasons.show()
    joinedDF.show()
    finalDF.show()
    solved.show()



if __name__ == "__main__":
    session = SparkSession.builder.getOrCreate()
    #ex_1(session)
    ex_2(session)

+--------+-----+-------------+------------+
|Username|TVSID|EpisodeNumber|SeasonNumber|
+--------+-----+-------------+------------+
|      U1| TVS1|            1|           1|
|      U1| TVS1|            1|           1|
|      U2| TVS2|            1|           1|
|      U2| TVS3|            1|           1|
|      U1| TVS4|            1|           1|
+--------+-----+-------------+------------+

+------+----------+
|TVSID2|MaxSeasons|
+------+----------+
|  TVS2|        11|
|  TVS3|         1|
|  TVS1|         2|
|  TVS4|         2|
+------+----------+

+--------+-----+----------+
|Username|TVSID|MaxSeasons|
+--------+-----+----------+
|      U2| TVS2|        11|
|      U2| TVS3|         1|
|      U1| TVS1|         2|
|      U1| TVS4|         2|
+--------+-----+----------+

+---------+-----------+
|Username2|MaxSeasons2|
+---------+-----------+
|       U2|         11|
|       U1|          2|
+---------+-----------+

+--------+-----+----------+---------+-----------+
|Username|TVSID|MaxSea